# 05 — Cascata de espacialização (preparação e passos 1 a 12)

**Objetivo:** preparar as bases de referência (Prefeitura e CNEFE) e localizar cada CNPJ pelos passos 1 a 12 da cascata (do mais preciso ao menos preciso). A Receita é a base que pode ter erros: são o CEP e o nome da via dela que são corrigidos; a Prefeitura é tentada antes do CNEFE.

**Entradas:** `df_estab` (01), `df_cnefe` e `df_base_cep` (02), `ref_pref` e `sirf_vias` (04).

**Saídas:** `03_processamento/05_cascata_espacializacao/`: `df_espacial` (resultado após o passo 12), `aux_estab`, `ref_cnefe`, `base_cep` e `ceps_nao_confiaveis`.

**Como rodar:** de cima para baixo. As funções dos passos estão em `src/cascata.py`; os limiares, em `parametros.cascata` do `config.yaml`.

**Seções:** 1. Preparação da cascata · 2 a 13. Passos 1 a 12 · 14. Gravação e validação · 15. Resumo

In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import datetime          # datas e horas
import importlib.util    # conferir se uma biblioteca está instalada, sem carregá-la
import json              # ler e gravar arquivos JSON
import os                # trocar arquivos de lugar com segurança
import re                # achar trechos de texto por padrão
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Bibliotecas de cálculos e de tabelas (DataFrames).
import numpy as np
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns: config.yaml, log, validação, arquivos de outras etapas.
from src import config, fontes, registro, validacao
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("05_cascata_espacializacao", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

# Funções e constantes do projeto usadas pelas seções (src/enderecos.py, src/cascata.py e src/geo.py).
from src.enderecos import (
    aplicar_nucleo, limpar_via, normalizar_numero, normalizar_texto, padronizar_bairro, tirar_numero_do_fim,
)
from src.cascata import (
    agrupar_por_cep, base_por_nucleo, centro_do_cep, combinar, gravar_resultado, interpolar,
    media_e_espalhamento, preparar_grupos, preparar_para_interpolar, resolver_exato_com_cep, resumo_passo,
    rodar_cadeia_nucleo, selecionar_pendentes,
)

from src import cascata

# Limiares das funções da cascata (src/cascata.py), lidos do config.yaml.
cascata.configurar(cfg)

# ===== LIMIARES DA CASCATA (config.yaml: parametros.cascata; ver docs/decisoes.md) =====
ESP_ESPALHADO = cfg.parametro("cascata.esp_espalhado")                  # até aqui, "espalhados"; acima, "muito espalhados"
ESP_MAX_VARIOS_CEPS = cfg.parametro("cascata.esp_max_varios_ceps")      # espalhamento máximo para aceitar via + número que existe em vários CEPs
LIMIAR_VIA_PARECIDA = cfg.parametro("cascata.limiar_via_parecida")      # semelhança mínima (0 a 1) de nomes de via "parecidos" (passo 2)
NOTA_MINIMA_FUZZY = cfg.parametro("cascata.nota_minima_fuzzy")          # passo 5: nota mínima (0 a 100) para aceitar
NOTA_MEDIA_FUZZY = cfg.parametro("cascata.nota_media_fuzzy")            # passo 5: nota mínima para a confiança ser "média"
EMPATE_MAX = cfg.parametro("cascata.empate_max")                        # empate: segunda via mais parecida até este número de pontos abaixo
NOTA_MINIMA_VIA_NO_CEP = cfg.parametro("cascata.nota_minima_via_no_cep")# passo 6: nota mínima para corrigir o nome da via no mesmo CEP
EXTENSAO_VIA_MAX = cfg.parametro("cascata.extensao_via_max")            # passo 8: extensão da via no CEP a partir da qual a confiança é "muito baixa"
MIN_CNPJ_CEP = cfg.parametro("cascata.min_cnpj_cep")                    # CEP não confiável: só avalia CEPs com pelo menos isso de CNPJs
MAX_CONCORDANCIA = cfg.parametro("cascata.max_concordancia")            # ... e concordância abaixo disso
MIN_NUCLEOS_DECLARADOS = cfg.parametro("cascata.min_nucleos_declarados")# ... e pelo menos isso de nomes de via diferentes declarados
NOTA_MINIMA_NUCLEO = cfg.parametro("cascata.nota_minima_nucleo")        # passo 10: nota mínima para aceitar um núcleo de nome parecido

# ===== ENTRADAS =====
df_estab = fontes.ler_processamento(cfg, "df_estab")          # 01
df_cnefe = fontes.ler_processamento(cfg, "df_cnefe")          # 02
df_base_cep = fontes.ler_processamento(cfg, "df_base_cep")    # 02
ref_pref = fontes.ler_processamento(cfg, "ref_pref")          # 04
sirf_vias = fontes.ler_processamento(cfg, "sirf_vias")        # 04
# CEP e via de todo o SIRF (usados para achar os CEPs não confiáveis, na seção 1).
cep_sirf = sirf_vias["cep"]
via_sirf = sirf_vias["via"]

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")

## 1. Preparação da cascata
Monta a base de endereços do CNEFE (`ref_cnefe`), a tabela auxiliar dos estabelecimentos da Receita (`aux_estab`, com via, número, núcleo do nome e bairro já tratados) e o `df_espacial` vazio, que os passos 1 a 15 vão preenchendo.
Prepara os agrupamentos das duas bases de referência (Prefeitura e CNEFE), pelo nome da via e pelo núcleo do nome, e por CEP (para o centro do CEP). Limpa a base de CEPs dos Correios e descobre os **CEPs não confiáveis** (CEPs "genéricos", como o 12227788): os que muitos CNPJs declaram com vias que nenhuma fonte conhece naquele CEP (concordância abaixo de 40%, com pelo menos 15 vias diferentes e 20 CNPJs). Esses CEPs nunca são usados sozinhos para localizar.
A regra geral da cascata: a Receita é a base que pode conter erros, então são o CEP e o nome da via dela que são corrigidos; a Prefeitura é tentada primeiro e o CNEFE depois.
Não altera nenhuma base lida, e não exibe linhas dos DataFrames, só logs.
As funções usadas pelos passos estão em `src/cascata.py`.

In [ ]:

log.info("SEÇÃO 1: preparação da cascata...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_b12 = time.time()

# ===================================================================
# ETAPA 1: BASE DE ENDEREÇOS DO CNEFE (ref_cnefe)
# ===================================================================
log.info("Etapa 1: montando a base de endereços do CNEFE (ref_cnefe)...")

# CEP sem espaços nas pontas.
cep_cnefe = df_cnefe["CEP"].fillna("").str.strip()
# Número normalizado (primeiro grupo de dígitos).
log.info("    Normalizando o número do endereço...")
num_cnefe = normalizar_numero(df_cnefe["NUM_ENDERECO"])
# Via normalizada: tipo + título (quando existe) + nome da via.
log.info("    Normalizando o nome da via...")
via_cnefe = normalizar_texto(
    df_cnefe["NOM_TIPO_SEGLOGR"].fillna("") + " "
    + df_cnefe["NOM_TITULO_SEGLOGR"].fillna("") + " "
    + df_cnefe["NOM_SEGLOGR"].fillna("")
)
# Aplica a limpeza completa da via (as mesmas regras das outras bases).
log.info("    Limpando o nome da via...")
via_cnefe = limpar_via(via_cnefe, "CNEFE")

# Tabela com uma linha por endereço do CNEFE.
ref_cnefe = pd.DataFrame({
    "cep": cep_cnefe.mask(cep_cnefe == ""),
    "via": via_cnefe.mask(via_cnefe == ""),
    "num": num_cnefe,
    # Latitude e longitude como número de verdade (o que não for número vira vazio).
    "lat": pd.to_numeric(df_cnefe["LATITUDE"], errors="coerce"),
    "lon": pd.to_numeric(df_cnefe["LONGITUDE"], errors="coerce"),
})

# Só entram as linhas que têm coordenada E via.
sem_coord_cnefe = ref_cnefe["lat"].isna() | ref_cnefe["lon"].isna()
sem_via_cnefe = ref_cnefe["via"].isna()
ref_cnefe = ref_cnefe[~sem_coord_cnefe & ~sem_via_cnefe].reset_index(drop=True)

log.info(f"Etapa 1 concluída! ref_cnefe: {len(ref_cnefe):,} linhas "
      f"({int(sem_coord_cnefe.sum()):,} sem coordenada e {int((~sem_coord_cnefe & sem_via_cnefe).sum()):,} sem via ficaram de fora).")

# ===================================================================
# ETAPA 2: TABELA AUXILIAR DOS ESTABELECIMENTOS (aux_estab)
# ===================================================================
log.info("\nEtapa 2: montando a tabela auxiliar dos estabelecimentos (aux_estab)...")

# Cria uma tabela nova com a MESMA numeração de linhas do df_estab (o df_estab não é alterado).
aux_estab = pd.DataFrame(index=df_estab.index)
# CNPJ de 14 dígitos, sem máscara: junta o básico, a ordem e o dígito verificador.
aux_estab["cnpj"] = df_estab["cnpj_basico"] + df_estab["cnpj_ordem"] + df_estab["cnpj_dv"]
# CEP, sem espaços nas pontas (vazio vira NaN).
cep_estab = df_estab["cep"].fillna("").str.strip()
aux_estab["cep"] = cep_estab.mask(cep_estab == "")
# Número normalizado (primeiro grupo de dígitos).
log.info("    Normalizando o número do endereço...")
aux_estab["num"] = normalizar_numero(df_estab["numero"])
# Via normalizada: tipo (RUA, AVENIDA...) + nome da via, em maiúsculas e sem acentos.
log.info("    Normalizando o nome da via...")
via_estab = normalizar_texto(df_estab["tipo_logradouro"].fillna("") + " " + df_estab["logradouro"].fillna(""))
# Tira do fim da via o número do próprio estabelecimento, quando ele aparece lá.
log.info("    Tirando o número do fim do nome da via...")
via_estab = tirar_numero_do_fim(via_estab, aux_estab["num"])
# Aplica a limpeza completa da via.
log.info("    Limpando o nome da via...")
via_estab = limpar_via(via_estab, "Receita")
aux_estab["via_limpa"] = via_estab.mask(via_estab == "")

log.info(f"Etapa 2 concluída! aux_estab: {len(aux_estab):,} linhas.")
log.info(f"    Sem número: {int(aux_estab['num'].isna().sum()):,} | "
      f"sem CEP: {int(aux_estab['cep'].isna().sum()):,} | "
      f"sem via: {int(aux_estab['via_limpa'].isna().sum()):,}")

# ===================================================================
# ETAPA 3: AGRUPAMENTOS DAS BASES DE REFERÊNCIA
# ===================================================================
log.info("\nEtapa 3: preparando os agrupamentos das bases de referência...")

# Dicionário com as duas bases, na ordem em que são tentadas: Prefeitura primeiro.
GRUPOS = {
    "Prefeitura": preparar_grupos(ref_pref, "Prefeitura"),
    "CNEFE": preparar_grupos(ref_cnefe, "CNEFE"),
}

log.info("Etapa 3 concluída!")
for nome_base, g in GRUPOS.items():
    log.info(f"    [{nome_base}] CEP+via+número: {len(g['cep_via_num']):,} | "
          f"via+número: {len(g['via_num']):,} | CEP+via: {len(g['cep_via']):,} | "
          f"vias distintas: {len(g['vias']):,}")

# ===================================================================
# ETAPA 4: df_espacial VAZIO
# ===================================================================
log.info("\nEtapa 4: criando o df_espacial vazio...")

# Mesma numeração de linhas do df_estab: a linha i do df_espacial é o estabelecimento i do df_estab.
df_espacial = pd.DataFrame(index=df_estab.index)
df_espacial["cnpj"] = aux_estab["cnpj"]
df_espacial["lat"] = np.nan                 # latitude do ponto
df_espacial["lon"] = np.nan                 # longitude do ponto
df_espacial["passo"] = np.nan               # passo da cascata que resolveu (1 a 8)
df_espacial["base"] = None                  # qual base deu o ponto (Prefeitura, CNEFE ou as duas)
df_espacial["metodo"] = None                # texto explicando COMO a coordenada foi obtida
df_espacial["confianca"] = None             # muito alta, alta, média, baixa ou muito baixa
df_espacial["espalhamento_m"] = np.nan      # espalhamento, em metros, dos pontos usados
df_espacial["distancia_bases_m"] = np.nan   # distância entre o ponto da Prefeitura e o do CNEFE
df_espacial["cep_usado"] = None             # CEP de fato usado (o da Receita ou o corrigido)
df_espacial["via_candidata"] = None         # via da base escolhida, quando o nome foi corrigido
df_espacial["nota_fuzzy"] = np.nan          # nota de 0 a 100 da semelhança entre os nomes de via
df_espacial["motivo_sem_coord"] = None      # por que o CNPJ ficou sem coordenada

log.info(f"Etapa 4 concluída! df_espacial: {len(df_espacial):,} linhas, {df_espacial.shape[1]} colunas "
      f"({'confere' if len(df_espacial) == len(df_estab) else 'NÃO confere'} com df_estab).")

# ===================================================================
# ETAPA 5: NÚCLEO DO NOME DA VIA E BAIRRO (passos 9 em diante)
# ===================================================================
log.info("\nEtapa 5: calculando o núcleo do nome das vias e o bairro padronizado...")

# Núcleo da via de cada estabelecimento (sem tipo, título e preposição).
aux_estab["nucleo"] = aplicar_nucleo(aux_estab["via_limpa"])
# Bairro da Receita, padronizado (JD vira JARDIM...).
aux_estab["bairro"] = padronizar_bairro(df_estab["bairro"])

# Mesmos agrupamentos da Etapa 3, mas pelo núcleo do nome: usados nos passos 9 e 10.
GRUPOS_NUCLEO = {
    "Prefeitura": preparar_grupos(base_por_nucleo(ref_pref), "Prefeitura, por núcleo"),
    "CNEFE": preparar_grupos(base_por_nucleo(ref_cnefe), "CNEFE, por núcleo"),
}

log.info("Etapa 5 concluída!")
log.info(f"    CNPJs com núcleo: {int(aux_estab['nucleo'].notna().sum()):,} | com bairro: {int(aux_estab['bairro'].notna().sum()):,}")
for nome_base, g in GRUPOS_NUCLEO.items():
    log.info(f"    [{nome_base}] núcleos distintos: {len(g['vias']):,}")

# ===================================================================
# ETAPA 6: BASE DE CEPs DOS CORREIOS JÁ LIMPA
# ===================================================================
log.info("\nEtapa 6: limpando a base de CEPs dos Correios...")

# O logradouro desta base traz extras: "Avenida Rui Barbosa - até 614/615" ou "Praça da Sé, 108".
# Tira tudo depois de " - " e tudo depois de ", número".
via_tb = (df_base_cep["LOGRADOURO"].fillna("")
          .str.replace(r"\s+-\s+.*$", "", regex=True)
          .str.replace(r",\s*\d.*$", "", regex=True))
# Mesma limpeza das outras bases (maiúsculas, sem acento, abreviações, números por extenso...).
via_tb = limpar_via(normalizar_texto(via_tb), "Base de CEPs")
# Tabela limpa: CEP, via, núcleo e bairro padronizado.
base_cep = pd.DataFrame({
    "cep": df_base_cep["CEP"].fillna("").str.strip(),
    "via": via_tb.mask(via_tb == ""),
    "bairro": padronizar_bairro(df_base_cep["BAIRRO"]),
})
base_cep["nucleo"] = aplicar_nucleo(base_cep["via"])

# Bairro de cada CEP (para desempatar vias com o mesmo nome em CEPs diferentes).
CEP_BAIRRO = base_cep.dropna(subset=["bairro"]).drop_duplicates("cep").set_index("cep")["bairro"].to_dict()
# CEPs de cada núcleo de via, segundo os Correios (passo 11).
CEPS_POR_NUCLEO_TB = base_cep.dropna(subset=["nucleo"]).groupby("nucleo")["cep"].apply(set).to_dict()
# As funções da cascata (src/cascata.py) usam o bairro de cada CEP para desempatar CEPs da mesma via.
cascata.definir_cep_bairro(CEP_BAIRRO)

log.info("Etapa 6 concluída!")
log.info(f"    CEPs com bairro conhecido: {len(CEP_BAIRRO):,} | núcleos de via na base de CEPs: {len(CEPS_POR_NUCLEO_TB):,}")

# ===================================================================
# ETAPA 7: CEPs NÃO CONFIÁVEIS (CEPs "GENÉRICOS", COMO O 12227788)
# ===================================================================
log.info("\nEtapa 7: procurando CEPs não confiáveis (concordância entre o CEP e a via declarada)...")

# Pares CEP + via conhecidos por alguma fonte: CNEFE, Prefeitura, SIRF inteiro (com e sem ponto) e Correios.
fontes_conhecidas = [
    ref_cnefe[["cep", "via"]],
    ref_pref[["cep", "via"]],
    pd.DataFrame({"cep": cep_sirf.mask(cep_sirf == ""), "via": via_sirf.mask(via_sirf == "")}),
    base_cep[["cep", "via"]],
]
conhecidos = pd.concat([f.dropna(subset=["cep", "via"]).drop_duplicates() for f in fontes_conhecidas],
                       ignore_index=True).drop_duplicates()
conhecidos["nucleo"] = aplicar_nucleo(conhecidos["via"])
conhecidos = conhecidos.dropna(subset=["nucleo"])[["cep", "nucleo"]].drop_duplicates()
conhecidos["conhece"] = True

# Para cada CNPJ: o par (CEP, núcleo) que ele declarou é conhecido em alguma fonte?
declarados = pd.DataFrame({"cep": aux_estab["cep"].values, "nucleo": aux_estab["nucleo"].values})
declarados = declarados.merge(conhecidos, on=["cep", "nucleo"], how="left")
declarados["conhece"] = declarados["conhece"].eq(True)

# Concordância de cada CEP: parte dos CNPJs que o declaram cuja via é conhecida naquele CEP.
por_cep = declarados[declarados["cep"].notna()].groupby("cep").agg(
    n_cnpj=("nucleo", "size"), n_concordam=("conhece", "sum"), n_nucleos=("nucleo", "nunique"))
por_cep["concordancia"] = por_cep["n_concordam"] / por_cep["n_cnpj"]
# CEP não confiável: muitos CNPJs, concordância baixa e muitas vias diferentes declaradas.
nao_confiaveis = por_cep[(por_cep["n_cnpj"] >= MIN_CNPJ_CEP)
                         & (por_cep["concordancia"] < MAX_CONCORDANCIA)
                         & (por_cep["n_nucleos"] >= MIN_NUCLEOS_DECLARADOS)].sort_values("n_cnpj", ascending=False)
CEPS_NAO_CONFIAVEIS = set(nao_confiaveis.index)

log.info("Etapa 7 concluída!")
log.info(f"    CEPs não confiáveis: {len(CEPS_NAO_CONFIAVEIS):,} (somam {int(nao_confiaveis['n_cnpj'].sum()):,} CNPJs)")
for cep_nc, linha in nao_confiaveis.iterrows():
    log.info(f"        {cep_nc}: {int(linha['n_cnpj']):,} CNPJs | concordância {linha['concordancia']:.0%} | "
          f"{int(linha['n_nucleos']):,} vias declaradas")

# ===================================================================
# ETAPA 8: AGRUPAMENTO POR CEP (CENTRO DO CEP, PASSOS 11 E 12)
# ===================================================================
log.info("\nEtapa 8: agrupando as bases de referência por CEP...")

GRUPOS_CEP = {"Prefeitura": agrupar_por_cep(ref_pref), "CNEFE": agrupar_por_cep(ref_cnefe)}

log.info("Etapa 8 concluída!")
for nome_base, g in GRUPOS_CEP.items():
    log.info(f"    [{nome_base}] CEPs com ponto: {len(g):,}")

log.info(f"\nSEÇÃO 1 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b12)}.")

## 2. Passo 1: CEP + número, via igual
Para cada estabelecimento com CEP e número, procura nas duas bases as linhas com o mesmo CEP, o mesmo número e o nome da via igual.
Quando as duas bases resolvem o mesmo estabelecimento e os pontos ficam a até 50 m, o método é "Prefeitura + CNEFE concordam" e a confiança é muito alta; quando ficam mais longe, vale o ponto da Prefeitura e o método registra que o CNEFE discorda.
A confiança de uma base só vem do espalhamento dos pontos: concentrados (alta), espalhados (média) ou muito espalhados (baixa).
Não altera as bases lidas, e não exibe linhas dos DataFrames, só logs.

In [ ]:
log.info("SEÇÃO 2: passo 1 (CEP + número, via igual)...")
inicio_bloco = time.time()
NOME_PASSO = "CEP + número, via igual"

# ETAPA 1: selecionar os estabelecimentos ainda sem coordenada.
log.info("Etapa 1: selecionando os estabelecimentos...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 1, exigir_num=True, exigir_cep=True)
log.info("Etapa 1 concluída!")

# ETAPA 2: procurar o endereço em cada base.
log.info("\nEtapa 2: procurando o endereço nas duas bases...")
resultados_base = {}
for nome_base, grupos in GRUPOS.items():
    log.info(f"    [{nome_base}] procurando...")
    resultados_base[nome_base] = resolver_exato_com_cep(est, grupos, parecida=False)
    log.info(f"    [{nome_base}] {len(resultados_base[nome_base]):,} estabelecimentos resolvidos.")
log.info("Etapa 2 concluída!")

# ETAPA 3: juntar as duas bases (com a regra de concordância).
log.info("\nEtapa 3: comparando as duas bases...")
resultado = combinar(resultados_base["Prefeitura"], resultados_base["CNEFE"], NOME_PASSO, concordancia=True)
# Neste passo o CEP usado é o da própria Receita (ele conferiu com a base).
if len(resultado):
    cep_por_idx = est.set_index("idx")["cep"]
    resultado["cep_usado"] = cep_por_idx.reindex(resultado.index)
log.info("Etapa 3 concluída!")

# ETAPA 4: gravar e resumir.
log.info("\nEtapa 4: gravando o resultado no df_espacial...")
gravar_resultado(df_espacial, resultado, 1)
log.info("Etapa 4 concluída!")
resumo_passo(df_espacial, 1, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 2 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 3. Passo 2: CEP + número, via parecida
Igual ao passo 1, mas para os estabelecimentos que sobraram: o CEP e o número batem com a base, e o nome da via é parecido (semelhança de 80% ou mais), o que trata os erros de digitação do cadastro da Receita.
Vale a mesma regra de concordância entre as duas bases e a mesma confiança pelo espalhamento. Guarda também a via escolhida e a nota de semelhança.
Não altera as bases lidas, e não exibe linhas dos DataFrames, só logs.

In [ ]:
log.info("SEÇÃO 3: passo 2 (CEP + número, via parecida)...")
inicio_bloco = time.time()
NOME_PASSO = "CEP + número, via parecida"

# ETAPA 1: selecionar os estabelecimentos ainda sem coordenada.
log.info("Etapa 1: selecionando os estabelecimentos...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 2, exigir_num=True, exigir_cep=True)
log.info("Etapa 1 concluída!")

# ETAPA 2: procurar o endereço em cada base, aceitando nome de via parecido.
log.info(f"\nEtapa 2: procurando nas duas bases (semelhança mínima de {LIMIAR_VIA_PARECIDA:.0%})...")
resultados_base = {}
for nome_base, grupos in GRUPOS.items():
    log.info(f"    [{nome_base}] procurando...")
    resultados_base[nome_base] = resolver_exato_com_cep(est, grupos, parecida=True)
    log.info(f"    [{nome_base}] {len(resultados_base[nome_base]):,} estabelecimentos resolvidos.")
log.info("Etapa 2 concluída!")

# ETAPA 3: juntar as duas bases (com a regra de concordância).
log.info("\nEtapa 3: comparando as duas bases...")
resultado = combinar(resultados_base["Prefeitura"], resultados_base["CNEFE"], NOME_PASSO, concordancia=True)
if len(resultado):
    cep_por_idx = est.set_index("idx")["cep"]
    resultado["cep_usado"] = cep_por_idx.reindex(resultado.index)
log.info("Etapa 3 concluída!")

# ETAPA 4: gravar e resumir.
log.info("\nEtapa 4: gravando o resultado no df_espacial...")
gravar_resultado(df_espacial, resultado, 2)
log.info("Etapa 4 concluída!")
resumo_passo(df_espacial, 2, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 3 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 4. Passo 3: número exato, via igual, CEP corrigido
Para quem sobrou, o CEP da Receita é considerado errado e ignorado: procura nas bases a mesma via com o mesmo número, em qualquer CEP do município.
Aceita quando a combinação existe em um único CEP (e aí esse CEP substitui o da Receita) ou, quando existe em vários CEPs, só se os pontos estiverem a até 50 m uns dos outros.
A Prefeitura é tentada primeiro; o CNEFE resolve o que ela não resolveu. A confiança nunca passa de média.
Não altera as bases lidas, e não exibe linhas dos DataFrames, só logs.

In [ ]:
log.info("SEÇÃO 4: passo 3 (número exato, via igual, CEP corrigido)...")
inicio_bloco = time.time()
NOME_PASSO = "número exato, via igual, CEP corrigido"

# ETAPA 1: selecionar os estabelecimentos ainda sem coordenada (o CEP não é exigido aqui).
log.info("Etapa 1: selecionando os estabelecimentos...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 3, exigir_num=True, exigir_cep=False)
log.info("Etapa 1 concluída!")


def resolver_via_num(est, grupos):
    """Procura a mesma via com o mesmo número, sem olhar o CEP da Receita."""
    cn = grupos["via_num"].rename(columns={"via": "via_e"})
    cruz = est.merge(cn, on=["via_e", "num"], how="inner")
    if cruz.empty:
        return pd.DataFrame()
    # Coordenada e espalhamento de cada combinação.
    cruz["lat"], cruz["lon"], cruz["espalhamento_m"] = media_e_espalhamento(cruz)
    # Um único CEP na base: não há ambiguidade. Vários CEPs: só com os pontos bem próximos.
    um_cep = cruz["ceps"] <= 1
    aceito = um_cep | (cruz["espalhamento_m"] <= ESP_MAX_VARIOS_CEPS)
    perto = cruz["espalhamento_m"] <= ESP_ESPALHADO
    cruz["sufixo"] = np.select(
        [um_cep & perto, um_cep],
        ["um CEP na base, pontos próximos", "um CEP na base, pontos espalhados"],
        default=f"vários CEPs na base, pontos a até {ESP_MAX_VARIOS_CEPS} m",
    )
    cruz["confianca"] = np.select([um_cep & perto, um_cep], ["média", "baixa"], default="média")
    # O CEP corrigido só é registrado quando a via + número existe em um CEP só.
    cruz["cep_usado"] = cruz["cep_min"].where(um_cep)
    res = cruz[aceito].set_index("idx")
    return res[["lat", "lon", "espalhamento_m", "sufixo", "confianca", "cep_usado"]]


# ETAPA 2: procurar em cada base.
log.info("\nEtapa 2: procurando via + número nas duas bases...")
resultados_base = {}
for nome_base, grupos in GRUPOS.items():
    log.info(f"    [{nome_base}] procurando...")
    resultados_base[nome_base] = resolver_via_num(est, grupos)
    log.info(f"    [{nome_base}] {len(resultados_base[nome_base]):,} estabelecimentos aceitos.")
log.info("Etapa 2 concluída!")

# ETAPA 3: juntar (Prefeitura primeiro, CNEFE depois).
log.info("\nEtapa 3: juntando as duas bases...")
resultado = combinar(resultados_base["Prefeitura"], resultados_base["CNEFE"], NOME_PASSO)
log.info("Etapa 3 concluída!")

# ETAPA 4: gravar e resumir.
log.info("\nEtapa 4: gravando o resultado no df_espacial...")
gravar_resultado(df_espacial, resultado, 3)
log.info("Etapa 4 concluída!")
resumo_passo(df_espacial, 3, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 4 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 5. Passo 4: interpolação, CEP + via igual
Para quem sobrou, o número exato não existe nas bases: procura na mesma via e no mesmo CEP os endereços com números vizinhos (o imediatamente menor e o imediatamente maior) e estima a posição proporcionalmente ao número.
Vizinhos da mesma paridade com diferença de até 50 números dão confiança média; os demais casos (diferença maior, paridade diferente ou um vizinho só) dão baixa.
A Prefeitura é tentada primeiro; o CNEFE resolve o que ela não resolveu.
Não altera as bases lidas, e não exibe linhas dos DataFrames, só logs.

In [ ]:
log.info("SEÇÃO 5: passo 4 (interpolação, CEP + via igual)...")
inicio_bloco = time.time()
NOME_PASSO = "interpolação, CEP + via igual"

# ETAPA 1: selecionar os estabelecimentos ainda sem coordenada.
log.info("Etapa 1: selecionando os estabelecimentos...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 4, exigir_num=True, exigir_cep=True)
# Deixa a tabela no formato que a interpolação exige (número de verdade e paridade).
est_i = preparar_para_interpolar(est)
log.info("Etapa 1 concluída!")

# Confiança de cada regra da interpolação neste passo.
CONFIANCAS_PASSO_4 = {"A_perto": "média", "A_longe": "baixa", "B": "baixa", "C": "baixa"}

# ETAPA 2: interpolar em cada base.
log.info("\nEtapa 2: interpolando nas duas bases...")
resultados_base = {}
for nome_base, grupos in GRUPOS.items():
    log.info(f"    [{nome_base}] interpolando...")
    resultados_base[nome_base] = interpolar(est_i, grupos["pontos"], ["cep", "via"], CONFIANCAS_PASSO_4)
    log.info(f"    [{nome_base}] {len(resultados_base[nome_base]):,} estabelecimentos resolvidos.")
log.info("Etapa 2 concluída!")

# ETAPA 3: juntar (Prefeitura primeiro, CNEFE depois).
log.info("\nEtapa 3: juntando as duas bases...")
resultado = combinar(resultados_base["Prefeitura"], resultados_base["CNEFE"], NOME_PASSO)
# O CEP usado é o da própria Receita (a interpolação foi feita dentro dele).
if len(resultado):
    cep_por_idx = est.set_index("idx")["cep"]
    resultado["cep_usado"] = cep_por_idx.reindex(resultado.index)
log.info("Etapa 3 concluída!")

# ETAPA 4: gravar e resumir.
log.info("\nEtapa 4: gravando o resultado no df_espacial...")
gravar_resultado(df_espacial, resultado, 4)
log.info("Etapa 4 concluída!")
resumo_passo(df_espacial, 4, NOME_PASSO, len(est_i), resultado)

log.info(f"\nSEÇÃO 5 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 6. Passo 5: número exato, via parecida, CEP corrigido
Para os estabelecimentos cuja via não existe com esse nome exato na base, procura a via de nome mais parecido entre as que têm o mesmo número, em qualquer CEP do município (o CEP da Receita é ignorado).
Aceita só com nota 90 ou mais, sem empate com outra via de nota quase igual e com os mesmos números (dígitos) nos dois nomes. Nota 95 ou mais dá confiança média; de 90 a 94, baixa.
A Prefeitura é tentada primeiro; o CNEFE resolve o que ela não resolveu. Exige a biblioteca `rapidfuzz`.
Não altera as bases lidas, e não exibe linhas dos DataFrames, só logs.

In [ ]:
# Biblioteca para procurar textos parecidos (precisa estar instalada: pip install rapidfuzz).
from rapidfuzz import process, fuzz

log.info("SEÇÃO 6: passo 5 (número exato, via parecida, CEP corrigido)...")
inicio_bloco = time.time()
NOME_PASSO = "número exato, via parecida, CEP corrigido"

# ETAPA 1: selecionar os estabelecimentos ainda sem coordenada (o CEP não é exigido aqui).
log.info("Etapa 1: selecionando os estabelecimentos...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 5, exigir_num=True, exigir_cep=False)
log.info("Etapa 1 concluída!")


def resolver_fuzzy_num(est, grupos, rotulo):
    """Acha a via de nome mais parecido entre as que têm o mesmo número na base."""
    # Só entram os estabelecimentos cuja via NÃO existe com esse nome exato na base.
    sub = est[~est["via_e"].isin(grupos["vias"])]
    log.info(f"        {len(sub):,} estabelecimentos com via inexistente nesta base.")
    if sub.empty:
        return pd.DataFrame()

    # Combinações distintas de via + número (evita repetir a mesma comparação).
    combos = sub[["via_e", "num"]].drop_duplicates().reset_index(drop=True)
    log.info(f"        comparando {len(combos):,} combinações via + número...")
    vias_por_num = grupos["vias_por_num"]

    melhores, notas, empates, digitos_iguais = [], [], [], []
    for i, (via, num) in enumerate(zip(combos["via_e"], combos["num"]), start=1):
        lista = vias_por_num.get(num)
        if not lista:
            # Nenhuma via da base tem este número: não há o que comparar.
            melhores.append(None); notas.append(0.0); empates.append(False); digitos_iguais.append(False)
        else:
            # extract devolve as 2 vias mais parecidas da lista, cada uma com sua nota (0 a 100).
            achados = process.extract(via, lista, scorer=fuzz.ratio, limit=2)
            melhor_via, melhor_nota = achados[0][0], achados[0][1]
            melhores.append(melhor_via)
            notas.append(melhor_nota)
            # "Empate": existe uma segunda via com nota quase igual à da melhor.
            empates.append(len(achados) > 1 and (melhor_nota - achados[1][1]) <= EMPATE_MAX)
            # Compara os números (dígitos) escritos nos dois nomes ("RUA 3 DE AGOSTO" tem ['3']).
            digitos_iguais.append(re.findall(r"\d+", via) == re.findall(r"\d+", melhor_via))
        if i % 5000 == 0:
            log.info(f"        {i:,} de {len(combos):,} combinações comparadas...")

    combos["via_c"] = melhores
    combos["score"] = notas
    combos["empate"] = empates
    combos["mesmos_digitos"] = digitos_iguais

    # Traz o resultado da comparação e, depois, os dados da via escolhida na base.
    t = sub.merge(combos, on=["via_e", "num"], how="left")
    t = t.merge(grupos["via_num"].rename(columns={"via": "via_c"}), on=["via_c", "num"], how="left")
    t["empate"] = t["empate"].astype(bool)
    t["mesmos_digitos"] = t["mesmos_digitos"].astype(bool)
    t = t[t["via_c"].notna() & t["n"].notna()]
    if t.empty:
        return pd.DataFrame()

    t["lat"], t["lon"], t["espalhamento_m"] = media_e_espalhamento(t)
    # Regras de aceitação.
    nota_ok = t["score"] >= NOTA_MINIMA_FUZZY
    ambigua = (t["ceps"] > 1) & (t["espalhamento_m"] > ESP_MAX_VARIOS_CEPS)
    aceito = nota_ok & ~t["empate"] & t["mesmos_digitos"] & ~ambigua
    log.info(f"        nota abaixo de {NOTA_MINIMA_FUZZY}: {int((~nota_ok).sum()):,} | "
          f"com empate: {int((nota_ok & t['empate']).sum()):,} | "
          f"dígitos diferentes: {int((nota_ok & ~t['empate'] & ~t['mesmos_digitos']).sum()):,} | "
          f"candidato ambíguo: {int(ambigua.sum()):,}")

    t["sufixo"] = np.where(t["score"] >= NOTA_MEDIA_FUZZY,
                           f"nota {NOTA_MEDIA_FUZZY} ou mais",
                           f"nota {NOTA_MINIMA_FUZZY} a {NOTA_MEDIA_FUZZY - 1}")
    t["confianca"] = np.where(t["score"] >= NOTA_MEDIA_FUZZY, "média", "baixa")
    t["cep_usado"] = t["cep_min"].where(t["ceps"] <= 1)
    t["via_candidata"] = t["via_c"]
    t["nota_fuzzy"] = t["score"]
    res = t[aceito].set_index("idx")
    return res[["lat", "lon", "espalhamento_m", "sufixo", "confianca", "cep_usado", "via_candidata", "nota_fuzzy"]]


# ETAPA 2: procurar em cada base.
log.info("\nEtapa 2: procurando a via mais parecida em cada base...")
resultados_base = {}
for nome_base, grupos in GRUPOS.items():
    log.info(f"    [{nome_base}] procurando...")
    resultados_base[nome_base] = resolver_fuzzy_num(est, grupos, nome_base)
    log.info(f"    [{nome_base}] {len(resultados_base[nome_base]):,} estabelecimentos aceitos.")
log.info("Etapa 2 concluída!")

# ETAPA 3: juntar (Prefeitura primeiro, CNEFE depois).
log.info("\nEtapa 3: juntando as duas bases...")
resultado = combinar(resultados_base["Prefeitura"], resultados_base["CNEFE"], NOME_PASSO)
log.info("Etapa 3 concluída!")

# ETAPA 4: gravar e resumir.
log.info("\nEtapa 4: gravando o resultado no df_espacial...")
gravar_resultado(df_espacial, resultado, 5)
log.info("Etapa 4 concluída!")
resumo_passo(df_espacial, 5, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 6 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 7. Passo 6: interpolação, via corrigida
Para quem sobrou, procura dentro do MESMO CEP da Receita a via de nome mais parecido (nota 85 ou mais, sem empate e com os mesmos números no nome) ou a via da base que aparece inteira no começo do nome da Receita (caso de complemento colado, como "RUA X (PARQUE Y)").
Com a via corrigida, interpola entre os números vizinhos daquele CEP. A confiança é sempre baixa, porque o nome da via foi trocado.
A Prefeitura é tentada primeiro; o CNEFE resolve o que ela não resolveu.
Não altera as bases lidas, e não exibe linhas dos DataFrames, só logs.

In [ ]:
log.info("SEÇÃO 7: passo 6 (interpolação, via corrigida)...")
inicio_bloco = time.time()
NOME_PASSO = "interpolação, via corrigida"

# ETAPA 1: selecionar os estabelecimentos ainda sem coordenada.
log.info("Etapa 1: selecionando os estabelecimentos...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 6, exigir_num=True, exigir_cep=True)
log.info("Etapa 1 concluída!")

# Confiança de cada regra da interpolação neste passo (todas baixas: o nome da via foi trocado).
CONFIANCAS_PASSO_6 = {"A_perto": "baixa", "A_longe": "baixa", "B": "baixa", "C": "baixa"}


def corrigir_via_no_cep(est, grupos):
    """Para cada par CEP + via da Receita, acha a via da base que provavelmente é a certa."""
    vias_por_cep = grupos["vias_por_cep"]
    # Pares distintos de CEP + via (evita repetir a mesma comparação).
    pares = est[["cep", "via_e"]].drop_duplicates().reset_index(drop=True)
    log.info(f"        comparando {len(pares):,} pares CEP + via...")

    corrigidas, notas = [], []
    for i, (cep, via) in enumerate(zip(pares["cep"], pares["via_e"]), start=1):
        lista = vias_por_cep.get(cep)
        # Sem CEP na base, ou via já existente nesse CEP, não há o que corrigir.
        if not lista or via in lista:
            corrigidas.append(None); notas.append(np.nan)
        else:
            # Regra 1: via da base (com 3 palavras ou mais) que aparece inteira no começo do nome da Receita.
            contidas = [v for v in lista if len(v.split()) >= 3 and via.startswith(v + " ")]
            if contidas:
                # A mais longa é a mais específica.
                corrigidas.append(max(contidas, key=len)); notas.append(100.0)
            else:
                # Regra 2: a via mais parecida do CEP, com nota alta, sem empate e com os mesmos dígitos.
                achados = process.extract(via, lista, scorer=fuzz.ratio, limit=2)
                melhor_via, melhor_nota = achados[0][0], achados[0][1]
                segunda = achados[1][1] if len(achados) > 1 else 0.0
                mesmos_digitos = re.findall(r"\d+", via) == re.findall(r"\d+", melhor_via)
                if melhor_nota >= NOTA_MINIMA_VIA_NO_CEP and (melhor_nota - segunda) > EMPATE_MAX and mesmos_digitos:
                    corrigidas.append(melhor_via); notas.append(melhor_nota)
                else:
                    corrigidas.append(None); notas.append(np.nan)
        if i % 5000 == 0:
            log.info(f"        {i:,} de {len(pares):,} pares comparados...")

    pares["via_c"] = corrigidas
    pares["nota_fuzzy"] = notas
    return pares[pares["via_c"].notna()]


# ETAPA 2: corrigir a via e interpolar, em cada base.
log.info("\nEtapa 2: corrigindo o nome da via e interpolando, nas duas bases...")
resultados_base = {}
for nome_base, grupos in GRUPOS.items():
    log.info(f"    [{nome_base}] corrigindo...")
    corrigidas = corrigir_via_no_cep(est, grupos)
    log.info(f"    [{nome_base}] {len(corrigidas):,} pares CEP + via corrigidos.")
    if corrigidas.empty:
        resultados_base[nome_base] = pd.DataFrame()
        continue
    # Troca o nome da via pelo corrigido e interpola dentro do mesmo CEP.
    t = est.merge(corrigidas, on=["cep", "via_e"], how="inner")
    t = t.drop(columns=["via_e"]).rename(columns={"via_c": "via_e"})
    t_i = preparar_para_interpolar(t)
    res = interpolar(t_i, grupos["pontos"], ["cep", "via"], CONFIANCAS_PASSO_6)
    if len(res):
        # Guarda a via escolhida e a nota, para você poder conferir depois.
        dados = t_i.set_index("idx")[["via", "nota_fuzzy"]]
        res["via_candidata"] = dados["via"].reindex(res.index)
        res["nota_fuzzy"] = dados["nota_fuzzy"].reindex(res.index)
    resultados_base[nome_base] = res
    log.info(f"    [{nome_base}] {len(res):,} estabelecimentos resolvidos.")
log.info("Etapa 2 concluída!")

# ETAPA 3: juntar (Prefeitura primeiro, CNEFE depois).
log.info("\nEtapa 3: juntando as duas bases...")
resultado = combinar(resultados_base["Prefeitura"], resultados_base["CNEFE"], NOME_PASSO)
if len(resultado):
    cep_por_idx = est.set_index("idx")["cep"]
    resultado["cep_usado"] = cep_por_idx.reindex(resultado.index)
log.info("Etapa 3 concluída!")

# ETAPA 4: gravar e resumir.
log.info("\nEtapa 4: gravando o resultado no df_espacial...")
gravar_resultado(df_espacial, resultado, 6)
log.info("Etapa 4 concluída!")
resumo_passo(df_espacial, 6, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 7 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 8. Passo 7: interpolação, CEP corrigido
Para quem sobrou, a via existe na base, mas não no CEP que a Receita informou. Quando essa via tem um único CEP na base, esse CEP substitui o da Receita e a interpolação é feita nele.
Se o par via + CEP também existir na outra base, a confiança é a mesma do passo 4; se só uma base indicar aquele CEP, a confiança é baixa.
A Prefeitura é tentada primeiro; o CNEFE resolve o que ela não resolveu.
Não altera as bases lidas, e não exibe linhas dos DataFrames, só logs.

In [ ]:
log.info("SEÇÃO 8: passo 7 (interpolação, CEP corrigido)...")
inicio_bloco = time.time()
NOME_PASSO = "interpolação, CEP corrigido"

# ETAPA 1: selecionar os estabelecimentos ainda sem coordenada (o CEP não é exigido aqui).
log.info("Etapa 1: selecionando os estabelecimentos...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 7, exigir_num=True, exigir_cep=False)
log.info("Etapa 1 concluída!")

# Confiança de cada regra da interpolação quando o CEP foi confirmado nas duas bases.
CONFIANCAS_PASSO_7 = {"A_perto": "média", "A_longe": "baixa", "B": "baixa", "C": "baixa"}

# ETAPA 2: trocar o CEP e interpolar, em cada base.
log.info("\nEtapa 2: procurando o CEP certo de cada via e interpolando...")
resultados_base = {}
for nome_base, grupos in GRUPOS.items():
    log.info(f"    [{nome_base}] procurando...")
    ceps_por_via = grupos["ceps_por_via"]
    # Para cada via da Receita, os CEPs que ela tem nesta base.
    vias_pendentes = pd.Series(est["via_e"].unique())
    # Fica só com as vias que existem na base e têm UM único CEP (sem ambiguidade).
    tabela_vias = pd.DataFrame({"via_e": vias_pendentes})
    tabela_vias["ceps"] = tabela_vias["via_e"].map(lambda v: ceps_por_via.get(v, set()))
    tabela_vias["qtd_ceps"] = tabela_vias["ceps"].map(len)
    com_um_cep = tabela_vias[tabela_vias["qtd_ceps"] == 1].copy()
    com_um_cep["cep_base"] = com_um_cep["ceps"].map(lambda s: next(iter(s)))
    log.info(f"    [{nome_base}] vias com um único CEP na base: {len(com_um_cep):,} "
          f"de {len(tabela_vias):,} vias pendentes.")
    if com_um_cep.empty:
        resultados_base[nome_base] = pd.DataFrame()
        continue

    # Troca o CEP da Receita pelo CEP da base.
    t = est.merge(com_um_cep[["via_e", "cep_base"]], on="via_e", how="inner")
    t = t.drop(columns=["cep"]).rename(columns={"cep_base": "cep"})
    t_i = preparar_para_interpolar(t)
    res = interpolar(t_i, grupos["pontos"], ["cep", "via"], CONFIANCAS_PASSO_7)
    if len(res):
        dados = t_i.set_index("idx")[["cep", "via"]]
        res["cep_usado"] = dados["cep"].reindex(res.index)
        # O CEP é "confirmado" quando o par CEP + via também existe na outra base.
        outra_base = "CNEFE" if nome_base == "Prefeitura" else "Prefeitura"
        pares_outra = GRUPOS[outra_base]["pares_cep_via"]
        confirmado = pd.Series(
            [(c, v) in pares_outra for c, v in zip(dados["cep"].reindex(res.index), dados["via"].reindex(res.index))],
            index=res.index,
        )
        # Sem confirmação na outra base, a confiança cai para baixa.
        res["confianca"] = np.where(confirmado, res["confianca"], "baixa")
        res["sufixo"] = res["sufixo"].astype(str) + np.where(
            confirmado, ", CEP confirmado nas duas bases", ", CEP de uma base só")
        log.info(f"    [{nome_base}] CEP confirmado nas duas bases: {int(confirmado.sum()):,} "
              f"de {len(res):,} resolvidos.")
    resultados_base[nome_base] = res
log.info("Etapa 2 concluída!")

# ETAPA 3: juntar (Prefeitura primeiro, CNEFE depois).
log.info("\nEtapa 3: juntando as duas bases...")
resultado = combinar(resultados_base["Prefeitura"], resultados_base["CNEFE"], NOME_PASSO)
log.info("Etapa 3 concluída!")

# ETAPA 4: gravar e resumir.
log.info("\nEtapa 4: gravando o resultado no df_espacial...")
gravar_resultado(df_espacial, resultado, 7)
log.info("Etapa 4 concluída!")
resumo_passo(df_espacial, 7, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 8 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 9. Passo 8: centro da via no CEP
Para quem sobrou (inclusive os sem número), usa o ponto central da via naquele CEP, ou seja, a média das coordenadas de todos os endereços da base com aquele CEP e aquela via.
A confiança é baixa quando a via naquele CEP tem até 1.000 m de extensão, e muito baixa acima disso. O espalhamento registrado é a própria extensão da via.
O motivo de quem ficou sem coordenada e o resumo final da cascata agora ficam no notebook 07 (seção 6), depois dos passos novos. Não altera as bases lidas, e não exibe linhas dos DataFrames, só logs.

In [ ]:
log.info("SEÇÃO 9: passo 8 (centro da via no CEP)...")
inicio_bloco = time.time()
NOME_PASSO = "centro da via no CEP"

# ETAPA 1: selecionar os estabelecimentos ainda sem coordenada (o número NÃO é exigido aqui).
log.info("Etapa 1: selecionando os estabelecimentos...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 8, exigir_num=False, exigir_cep=True)
# Guarda se o estabelecimento tem número (entra no nome do método).
est["tem_num"] = est["num"].notna()
log.info("Etapa 1 concluída!")


def resolver_centro_via(est, grupos):
    """Usa o ponto central da via naquele CEP."""
    cn = grupos["cep_via"].rename(columns={"via": "via_e"})
    cruz = est.merge(cn, on=["cep", "via_e"], how="inner")
    if cruz.empty:
        return pd.DataFrame()
    cruz["lat"], cruz["lon"], cruz["espalhamento_m"] = media_e_espalhamento(cruz)
    # Extensão da via naquele CEP: quanto maior, menos o ponto central significa.
    via_curta = cruz["espalhamento_m"] <= EXTENSAO_VIA_MAX
    cruz["sufixo"] = (np.where(cruz["tem_num"], "com número", "sem número")
                      + np.where(via_curta, f", via até {EXTENSAO_VIA_MAX} m",
                                 f", via acima de {EXTENSAO_VIA_MAX} m"))
    cruz["confianca"] = np.where(via_curta, "baixa", "muito baixa")
    res = cruz.set_index("idx")
    return res[["lat", "lon", "espalhamento_m", "sufixo", "confianca"]]


# ETAPA 2: procurar em cada base.
log.info("\nEtapa 2: calculando o centro da via nas duas bases...")
resultados_base = {}
for nome_base, grupos in GRUPOS.items():
    log.info(f"    [{nome_base}] calculando...")
    resultados_base[nome_base] = resolver_centro_via(est, grupos)
    log.info(f"    [{nome_base}] {len(resultados_base[nome_base]):,} estabelecimentos resolvidos.")
log.info("Etapa 2 concluída!")

# ETAPA 3: juntar (Prefeitura primeiro, CNEFE depois).
log.info("\nEtapa 3: juntando as duas bases...")
resultado = combinar(resultados_base["Prefeitura"], resultados_base["CNEFE"], NOME_PASSO)
if len(resultado):
    cep_por_idx = est.set_index("idx")["cep"]
    resultado["cep_usado"] = cep_por_idx.reindex(resultado.index)
log.info("Etapa 3 concluída!")

# ETAPA 4: gravar.
log.info("\nEtapa 4: gravando o resultado no df_espacial...")
gravar_resultado(df_espacial, resultado, 8)
log.info("Etapa 4 concluída!")
resumo_passo(df_espacial, 8, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 9 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 10. Passo 9: núcleo do nome igual
Compara pelo **núcleo do nome** (sem tipo, títulos e preposições), o que resolve casos como "AVENIDA JORGE ZARUR" × "AVENIDA DOUTOR JORGE ZARUR", "AVENIDA BACABAL" × "RUA BACABAL" e "RUA 15 DE JULHO" × "RUA 15 DE JULHO DIA DO COMERCIANTE"... desde que o núcleo seja o mesmo.
Tenta, nesta ordem, e em cada método a Prefeitura primeiro e o CNEFE depois: (1) CEP + número; (2) interpolação no CEP; (3) número exato em outro CEP; (4) interpolação no CEP corrigido (a via tem um CEP só na base, ou o bairro da base de CEPs desempata); (5) interpolação na via inteira, ignorando o CEP, só com vizinhos dos dois lados a até 300 m (avenidas longas com vários CEPs); (6) centro da via no CEP; (7) centro da via no CEP corrigido.
A confiança segue as mesmas regras dos passos 1 a 8 e cai nos métodos com CEP corrigido. Não altera as bases lidas, só logs.

In [ ]:
log.info("SEÇÃO 10: passo 9 (núcleo do nome igual)...")
inicio_bloco = time.time()
NOME_PASSO = "núcleo do nome igual"

# ETAPA 1: selecionar os estabelecimentos ainda sem coordenada (número e CEP não são exigidos aqui).
log.info("Etapa 1: selecionando os estabelecimentos...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 9, exigir_num=False, exigir_cep=False, coluna_via="nucleo", incluir_bairro=True)
log.info("Etapa 1 concluída!")

# ETAPA 2: tentar os métodos em ordem, do mais preciso para o menos preciso.
log.info("\nEtapa 2: tentando os métodos pelo núcleo do nome (Prefeitura primeiro, CNEFE depois)...")
resultado = rodar_cadeia_nucleo(GRUPOS_NUCLEO, est, NOME_PASSO)
log.info("Etapa 2 concluída!")

# ETAPA 3: gravar e resumir.
log.info("\nEtapa 3: gravando o resultado no df_espacial...")
gravar_resultado(df_espacial, resultado, 9)
log.info("Etapa 3 concluída!")
resumo_passo(df_espacial, 9, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 10 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 11. Passo 10: núcleo do nome corrigido (iniciais e grafia parecida)
Para quem tem um núcleo que não existe em nenhuma base, procura o núcleo certo: primeiro por **iniciais** ("JOAO B NOGUEIRA" = "JOAO BATISTA NOGUEIRA", só quando há um único candidato), depois por **grafia parecida** (nota mínima 90, sem empate e com os mesmos números escritos no nome, como "KUBITSCHECK" × "KUBITSCHEK").
Com o núcleo corrigido, roda os mesmos métodos do passo 9. A confiança fica **no máximo baixa**, e o núcleo escolhido e a nota ficam em `via_candidata` e `nota_fuzzy`. Exige a biblioteca `rapidfuzz`. Só logs.

In [ ]:
# Biblioteca para procurar textos parecidos (precisa estar instalada: pip install rapidfuzz).
from rapidfuzz import process, fuzz
# Dicionário que cria listas vazias sozinho (usado no índice das iniciais).
from collections import defaultdict

log.info("SEÇÃO 11: passo 10 (núcleo do nome corrigido: iniciais e grafia parecida)...")
inicio_bloco = time.time()
NOME_PASSO = "núcleo do nome corrigido"

# ETAPA 1: selecionar os estabelecimentos ainda sem coordenada cujo núcleo NÃO existe nas bases.
log.info("Etapa 1: selecionando os estabelecimentos...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 10, exigir_num=False, exigir_cep=False, coluna_via="nucleo", incluir_bairro=True)
# Todos os núcleos das duas bases (os candidatos à correção), em ordem.
nucleos_base = sorted(GRUPOS_NUCLEO["Prefeitura"]["vias"] | GRUPOS_NUCLEO["CNEFE"]["vias"])
# Quem tem um núcleo que já existe nas bases não precisa de correção (já passou pelo passo 9).
est = est[~est["via_e"].isin(set(nucleos_base))].reset_index(drop=True)
log.info(f"    Com núcleo inexistente nas bases: {len(est):,} estabelecimentos "
      f"({est['via_e'].nunique():,} núcleos diferentes).")
log.info("Etapa 1 concluída!")


def iniciais_compativeis(a, b):
    """
    Verdadeiro quando os dois nomes têm as mesmas palavras, na mesma ordem, a não ser por iniciais.
    Exemplo: 'JOAO B NOGUEIRA' e 'JOAO BATISTA NOGUEIRA'. Exige pelo menos uma inicial.
    """
    pa, pb = a.split(), b.split()
    if len(pa) != len(pb) or len(pa) < 2:
        return False
    tem_inicial = False
    for x, y in zip(pa, pb):
        if x == y:
            continue
        # Uma das palavras é uma letra só e é a primeira letra da outra.
        if (len(x) == 1 and y.startswith(x)) or (len(y) == 1 and x.startswith(y)):
            tem_inicial = True
            continue
        return False
    return tem_inicial


def corrigir_nucleos(pendentes_nucleos, candidatos):
    """
    Para cada núcleo pendente, procura o núcleo certo entre os candidatos das bases:
    regra 1, iniciais (um só candidato compatível); regra 2, grafia parecida (nota alta,
    sem empate e com os mesmos números escritos no nome). Devolve uma tabela com as correções aceitas.
    """
    # Índice dos candidatos por (quantidade de palavras, primeira palavra, última palavra): deixa a regra 1 rápida.
    indice = defaultdict(list)
    for c in candidatos:
        palavras = c.split()
        indice[(len(palavras), palavras[0], palavras[-1])].append(c)

    linhas = []
    for i, p in enumerate(pendentes_nucleos, start=1):
        palavras = p.split()
        # Regra 1: iniciais.
        compativeis = [c for c in indice.get((len(palavras), palavras[0], palavras[-1]), []) if iniciais_compativeis(p, c)]
        if len(compativeis) == 1:
            linhas.append((p, compativeis[0], 100.0, "iniciais"))
        elif len(compativeis) == 0:
            # Regra 2: grafia parecida (as 2 melhores notas, de 0 a 100).
            achados = process.extract(p, candidatos, scorer=fuzz.ratio, limit=2)
            if achados:
                melhor, nota = achados[0][0], achados[0][1]
                segunda = achados[1][1] if len(achados) > 1 else 0.0
                mesmos_digitos = re.findall(r"\d+", p) == re.findall(r"\d+", melhor)
                if nota >= NOTA_MINIMA_NUCLEO and (nota - segunda) > EMPATE_MAX and mesmos_digitos:
                    linhas.append((p, melhor, round(float(nota), 1), "grafia parecida"))
        # Com mais de um candidato de iniciais compatíveis: ambíguo, não corrige.
        if i % 500 == 0:
            log.info(f"        {i:,} de {len(pendentes_nucleos):,} núcleos comparados...")
    return pd.DataFrame(linhas, columns=["via_e", "nucleo_corrigido", "nota_fuzzy", "regra"])


# ETAPA 2: corrigir os núcleos.
log.info("\nEtapa 2: procurando o núcleo certo de cada via (iniciais e grafia parecida)...")
correcoes = corrigir_nucleos(list(est["via_e"].unique()), nucleos_base)
log.info("Etapa 2 concluída!")
for nome_regra, qtd in correcoes["regra"].value_counts().items():
    log.info(f"    Núcleos corrigidos por {nome_regra}: {qtd:,}")

# ETAPA 3: rodar os mesmos métodos do passo 9 com o núcleo corrigido (confiança no máximo "baixa").
log.info("\nEtapa 3: tentando os métodos com o núcleo corrigido...")
t = est.merge(correcoes, on="via_e", how="inner")
t = t.drop(columns=["via_e"]).rename(columns={"nucleo_corrigido": "via_e"})
resultado = rodar_cadeia_nucleo(GRUPOS_NUCLEO, t, NOME_PASSO, teto="baixa")
if len(resultado):
    dados = t.set_index("idx")
    resultado["via_candidata"] = dados["via_e"].reindex(resultado.index)
    resultado["nota_fuzzy"] = dados["nota_fuzzy"].reindex(resultado.index)
    resultado["metodo"] = resultado["metodo"].astype(str) + " (" + dados["regra"].reindex(resultado.index).astype(str) + ")"
log.info("Etapa 3 concluída!")

# ETAPA 4: gravar e resumir.
log.info("\nEtapa 4: gravando o resultado no df_espacial...")
gravar_resultado(df_espacial, resultado, 10)
log.info("Etapa 4 concluída!")
resumo_passo(df_espacial, 10, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 11 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 12. Passo 11: via encontrada só na base de CEPs
Para quem sobrou, procura o núcleo da via na base de CEPs dos Correios e escolhe o CEP: o da própria Receita, se ele for um dos CEPs da via; senão, o único CEP da via; senão, o que tem o mesmo bairro. CEPs não confiáveis nunca são escolhidos.
O ponto é o **centro desse CEP** (média dos endereços das bases com aquele CEP), só quando o CEP é compacto: até 300 m de espalhamento, confiança baixa; de 301 a 500 m, muito baixa; acima disso, não usa. Só logs.

In [ ]:
log.info("SEÇÃO 12: passo 11 (via encontrada só na base de CEPs: centro do CEP dela)...")
inicio_bloco = time.time()
NOME_PASSO = "CEP achado pela via na base de CEPs, centro do CEP"

# ETAPA 1: selecionar os estabelecimentos ainda sem coordenada.
log.info("Etapa 1: selecionando os estabelecimentos...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 11, exigir_num=False, exigir_cep=False, coluna_via="nucleo", incluir_bairro=True)
log.info("Etapa 1 concluída!")

# ETAPA 2: achar o CEP de cada via pela base de CEPs dos Correios.
log.info("\nEtapa 2: procurando o CEP de cada via na base de CEPs...")
ceps_escolhidos, motivos_cep = [], []
for via, bairro, cep_receita in zip(est["via_e"], est["bairro"], est["cep"]):
    # CEPs da via segundo os Correios, sem os CEPs não confiáveis.
    candidatos = sorted(CEPS_POR_NUCLEO_TB.get(via, set()) - CEPS_NAO_CONFIAVEIS)
    if cep_receita in candidatos:
        # O CEP da Receita é um dos CEPs da via: fica confirmado.
        ceps_escolhidos.append(cep_receita); motivos_cep.append("CEP da Receita confirmado pela base de CEPs")
    elif len(candidatos) == 1:
        ceps_escolhidos.append(candidatos[0]); motivos_cep.append("via com um CEP só na base de CEPs")
    elif len(candidatos) > 1 and isinstance(bairro, str) and \
            len([c for c in candidatos if CEP_BAIRRO.get(c) == bairro]) == 1:
        ceps_escolhidos.append([c for c in candidatos if CEP_BAIRRO.get(c) == bairro][0])
        motivos_cep.append("CEP desempatado pelo bairro")
    else:
        ceps_escolhidos.append(None); motivos_cep.append(None)
t = est.assign(cep=ceps_escolhidos, motivo_cep=motivos_cep)
t = t[t["cep"].notna()]
log.info(f"Etapa 2 concluída! CEP achado para {len(t):,} de {len(est):,} estabelecimentos.")
for nome_motivo, qtd in t["motivo_cep"].value_counts().items():
    log.info(f"    {nome_motivo}: {qtd:,}")

# ETAPA 3: centro do CEP em cada base (só CEP compacto).
log.info("\nEtapa 3: calculando o centro do CEP nas duas bases...")
resultados_base = {}
for nome_base, grupos_cep in GRUPOS_CEP.items():
    log.info(f"    [{nome_base}] calculando...")
    res = centro_do_cep(t, grupos_cep, "centro do CEP") if len(t) else pd.DataFrame()
    if len(res):
        res["sufixo"] = res["sufixo"].astype(str) + " (" + t.set_index("idx")["motivo_cep"].reindex(res.index).astype(str) + ")"
    resultados_base[nome_base] = res
    log.info(f"    [{nome_base}] {len(res):,} estabelecimentos resolvidos.")
log.info("Etapa 3 concluída!")

# ETAPA 4: juntar (Prefeitura primeiro, CNEFE depois), gravar e resumir.
log.info("\nEtapa 4: juntando as duas bases e gravando o resultado no df_espacial...")
resultado = combinar(resultados_base["Prefeitura"], resultados_base["CNEFE"], NOME_PASSO)
if len(resultado):
    # Guarda a via usada para achar o CEP (o núcleo).
    resultado["via_candidata"] = t.set_index("idx")["via_e"].reindex(resultado.index)
gravar_resultado(df_espacial, resultado, 11)
log.info("Etapa 4 concluída!")
resumo_passo(df_espacial, 11, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 12 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 13. Passo 12: centro do CEP da Receita
Para quem sobrou e tem um CEP **confiável**, usa o centro desse CEP, com as mesmas regras do passo 11 (só CEP compacto; confiança baixa até 300 m e muito baixa de 301 a 500 m). CNPJs com CEP não confiável seguem para o geocoding. Só logs.

In [ ]:
log.info("SEÇÃO 13: passo 12 (centro do CEP da Receita)...")
inicio_bloco = time.time()
NOME_PASSO = "centro do CEP da Receita"

# ETAPA 1: selecionar os estabelecimentos ainda sem coordenada, com CEP (a via não é exigida aqui).
log.info("Etapa 1: selecionando os estabelecimentos...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 12, exigir_num=False, exigir_cep=True, exigir_via=False)
# CEP não confiável não serve para localizar: esses CNPJs seguem para o geocoding.
nao_confiavel = est["cep"].isin(CEPS_NAO_CONFIAVEIS)
log.info(f"    Com CEP não confiável (seguem para o próximo passo): {int(nao_confiavel.sum()):,}")
est = est[~nao_confiavel]
log.info("Etapa 1 concluída!")

# ETAPA 2: centro do CEP em cada base (só CEP compacto).
log.info("\nEtapa 2: calculando o centro do CEP nas duas bases...")
resultados_base = {}
for nome_base, grupos_cep in GRUPOS_CEP.items():
    log.info(f"    [{nome_base}] calculando...")
    resultados_base[nome_base] = centro_do_cep(est, grupos_cep, "centro do CEP") if len(est) else pd.DataFrame()
    log.info(f"    [{nome_base}] {len(resultados_base[nome_base]):,} estabelecimentos resolvidos.")
log.info("Etapa 2 concluída!")

# ETAPA 3: juntar (Prefeitura primeiro, CNEFE depois), gravar e resumir.
log.info("\nEtapa 3: juntando as duas bases e gravando o resultado no df_espacial...")
resultado = combinar(resultados_base["Prefeitura"], resultados_base["CNEFE"], NOME_PASSO)
gravar_resultado(df_espacial, resultado, 12)
log.info("Etapa 3 concluída!")
resumo_passo(df_espacial, 12, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 13 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 14. Gravação e validação
Confere que há uma linha por estabelecimento e grava o resultado e as tabelas usadas nos notebooks 06 e 07.

In [ ]:
log.info("SEÇÃO 14: gravação e validação...")
# Uma linha por estabelecimento, na mesma ordem do df_estab.
assert len(df_espacial) == len(df_estab), "df_espacial precisa ter uma linha por estabelecimento"
validacao.validar_tabela("df_espacial", df_espacial, chave="cnpj", obrigatorias=["cnpj"], linhas_entrada=len(df_estab))
# CEPs não confiáveis: o CEP vira coluna (era o índice da tabela).
ceps_nao_confiaveis = nao_confiaveis.reset_index()
fontes.gravar_processamento(cfg, "espacial_05", df_espacial)
fontes.gravar_processamento(cfg, "aux_estab", aux_estab)
fontes.gravar_processamento(cfg, "ref_cnefe", ref_cnefe)
fontes.gravar_processamento(cfg, "base_cep", base_cep)
fontes.gravar_processamento(cfg, "ceps_nao_confiaveis", ceps_nao_confiaveis)
log.info("SEÇÃO 14 concluída!")

## 15. Resumo
Números principais, arquivos gerados e tempo total.

In [ ]:
log.info("RESUMO do 05_cascata_espacializacao...")
com_ponto = int(df_espacial["lat"].notna().sum())
log.info(f"Com coordenada após o passo 12: {com_ponto:,} de {len(df_espacial):,} ({com_ponto / len(df_espacial) * 100:.1f}%)")
log.info("Arquivos gerados:")
for chave in ['espacial_05', 'aux_estab', 'ref_cnefe', 'base_cep', 'ceps_nao_confiaveis']:
    caminho = cfg.caminho("processamento." + chave)
    log.info(f"    {caminho.relative_to(RAIZ)} ({caminho.stat().st_size / 1e6:.1f} MB)")
log.info(f"RESUMO concluído! Tempo total do notebook: {formatar_tempo(time.time() - inicio_notebook)}.")